# Create flag parameter

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
# creating utilities

dbutils.widgets.text('incremental_flag', '0')

In [0]:
# check if the table exists with any data then 
if spark.catalog.tableExists('cars_catalog.gold.dim_dealer'):
    if (spark.sql("SELECT MAX(dim_dealer_key) FROM cars_catalog.gold.dim_dealer").collect()[0][0] > 0):
        incremental_flag = '1'
else:
    incremental_flag = dbutils.widgets.get('incremental_flag')

print(incremental_flag)

1


# creating dimension dealer


### Fetch relative columns

In [0]:
# spark.sql('''
#                       SELECT *
#                       FROM parquet.`abfss://silver@adlsforde.dfs.core.windows.net/carsales`
#                       ''').display()

In [0]:
source_df = spark.sql('''
                      SELECT DISTINCT(DEALER_ID) as dealer_id, DEALERNAME As dealer_name
                      FROM parquet.`abfss://silver@adlsforde.dfs.core.windows.net/carsales`
                      ''')



In [0]:
source_df.display()

dealer_id,dealer_name
DLR0069,Geo Motors
DLR0249,Acura Motors
DLR0209,Zastava Motors
DLR0189,Sunbeam Motors
DLR0130,Micro Motors
DLR0093,Iso Motors
DLR0135,Morgan Motors
DLR0170,Saleen Motors
DLR0265,Bentley Motors
DLR0087,Hyundai Motors



### Dim Dealer Sink initial and incremental (Just Bring the schema if table not exitsis)

In [0]:
if spark.catalog.tableExists('cars_catalog.gold.dim_dealer'):
    sink_df = spark.sql('''
                        SELECT dim_dealer_key, dealer_id, dealer_name
                        FROM cars_catalog.gold.dim_dealer
                        ''')
# Initial dimention table creation
else:
    sink_df = spark.sql('''
                        SELECT 1 as dim_dealer_key, DEALER_ID as dealer_id, DEALERNAME as dealer_name
                    FROM parquet.`abfss://silver@adlsforde.dfs.core.windows.net/carsales`
                    WHERE 1 = 0
                    ''')


### Fintering new records and old records

In [0]:
filter_df = source_df.join(sink_df, source_df.dealer_id == sink_df.dealer_id, 'left')\
    .select(source_df.dealer_id, source_df.dealer_name, sink_df.dim_dealer_key)

**df_filter_old**

In [0]:
df_filter_old = filter_df.filter(col('dim_dealer_key').isNotNull())


**df_filter_new**

In [0]:
df_filter_new = filter_df.filter(col('dim_dealer_key').isNull()).select('dealer_id', 'dealer_name')


### Create Surrogate Key
**Fetch the max surrogate key from existing dim table**

In [0]:
if (incremental_flag == '0'):
    max_value = 1
else:
    max_value_df = spark.sql("SELECT MAX(dim_dealer_key) FROM cars_catalog.gold.dim_dealer")
    max_value = max_value_df.collect()[0][0]




**Create Surrogate key column and add the max surrogate key**

In [0]:
df_filter_new = df_filter_new.withColumn('dim_dealer_key', max_value + monotonically_increasing_id())
df_filter_new.display()

dealer_id,dealer_name,dim_dealer_key


### Create Final data frame - df_filter_old + df_filter_new


In [0]:
final_df =  df_filter_new.union(df_filter_old)

In [0]:
final_df.display()

dealer_id,dealer_name,dim_dealer_key
DLR0069,Geo Motors,1
DLR0249,Acura Motors,2
DLR0209,Zastava Motors,3
DLR0189,Sunbeam Motors,4
DLR0130,Micro Motors,5
DLR0093,Iso Motors,6
DLR0135,Morgan Motors,7
DLR0170,Saleen Motors,8
DLR0265,Bentley Motors,9
DLR0087,Hyundai Motors,10


### SCD Type - 1 (UPSERT)
**Update(existing change data) + Insert(new data)**

In [0]:
# Incremental Run
if spark.catalog.tableExists("cars_catalog.gold.dim_dealer"):
    delta_table = DeltaTable.forPath(spark, "abfss://gold@adlsforde.dfs.core.windows.net/dim_dealer")
    
    delta_table.alias('trg').merge(final_df.alias('src'), "trg.dim_dealer_key = src.dim_dealer_key")\
                            .whenMatchedUpdateAll()\
                            .whenNotMatchedInsertAll()\
                            .execute()
# initial run
else:
    final_df.write.format('delta')\
        .mode("overwrite")\
        .option("path", "abfss://gold@adlsforde.dfs.core.windows.net/dim_dealer")\
        .saveAsTable("cars_catalog.gold.dim_dealer")

In [0]:
%sql
SELECT * FROM cars_catalog.gold.dim_dealer

dealer_id,dealer_name,dim_dealer_key
DLR0069,Geo Motors,1
DLR0249,Acura Motors,2
DLR0209,Zastava Motors,3
DLR0189,Sunbeam Motors,4
DLR0130,Micro Motors,5
DLR0093,Iso Motors,6
DLR0135,Morgan Motors,7
DLR0170,Saleen Motors,8
DLR0265,Bentley Motors,9
DLR0087,Hyundai Motors,10
